In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from functools import reduce


# CONFIG
DATA_DIR = "/user/data/raw"
FILE_PREFIX = "yellow_tripdata"
START_YEAR = 2020
END_YEAR = 2025
SLOT_MINUTES = 30

spark = (
    SparkSession.builder
    .appName("TaxiEDA_Compact")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.default.parallelism", "64")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .getOrCreate()
)

# HELPERS
def build_file_paths(data_dir, file_prefix, start_year, end_year):
    return [
        f"{data_dir}/{file_prefix}_{y}-{m:02d}.parquet"
        for y in range(start_year, end_year + 1)
        for m in range(1, 13)
    ]

def summarize_files_quiet(spark, paths):
    records = []

    for p in paths:
        try:
            tmp = spark.read.parquet(p).select(
                F.col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
                F.col("PULocationID").cast("int").alias("PULocationID")
            )

            n_rows = tmp.count()
            mm = tmp.select(
                F.min("tpep_pickup_datetime").alias("min_time"),
                F.max("tpep_pickup_datetime").alias("max_time"),
                F.countDistinct("PULocationID").alias("n_zones")
            ).collect()[0]

            records.append((
                p, True, n_rows, mm["min_time"], mm["max_time"], mm["n_zones"], None
            ))
        except Exception as e:
            records.append((p, False, None, None, None, None, str(e)))

    schema = """
        source_file string,
        read_ok boolean,
        n_rows long,
        min_time timestamp,
        max_time timestamp,
        n_zones long,
        error string
    """
    return spark.createDataFrame(records, schema=schema)

def read_union_safely(spark, file_summary):
    ok_paths = [
        r["source_file"]
        for r in file_summary.filter(F.col("read_ok") == True).select("source_file").collect()
    ]

    dfs = []
    for p in ok_paths:
        tmp = spark.read.parquet(p).select(
            F.col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
            F.col("PULocationID").cast("int").alias("PULocationID"),
            F.input_file_name().alias("source_file")
        )
        dfs.append(tmp)

    if not dfs:
        raise ValueError("Không có file nào đọc được.")

    return reduce(lambda a, b: a.unionByName(b), dfs)


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f0f4f42c-c470-41a3-b171-7a559054b7db;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 141ms :: artifacts dl 6ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

In [2]:
import pandas as pd

ONE_FILE_PATH = "/user/data/raw/yellow_tripdata_2025-01.parquet"

df_one = spark.read.parquet(ONE_FILE_PATH)

schema_table = pd.DataFrame([
    {
        "Column Name": field.name,
        "Data Type": str(field.dataType),
        "Nullable": field.nullable
    }
    for field in df_one.schema.fields
])

schema_table

,Column Name,Data Type,Nullable
0,VendorID,IntegerType(),True
1,tpep_pickup_datetime,TimestampNTZType(),True
2,tpep_dropoff_datetime,TimestampNTZType(),True
3,passenger_count,LongType(),True
4,trip_distance,DoubleType(),True
5,RatecodeID,LongType(),True
6,store_and_fwd_flag,StringType(),True
7,PULocationID,IntegerType(),True
8,DOLocationID,IntegerType(),True
9,payment_type,LongType(),True


In [4]:
# 1) FILE-LEVEL SUMMARY
paths = build_file_paths(DATA_DIR, FILE_PREFIX, START_YEAR, END_YEAR)
file_summary = summarize_files_quiet(spark, paths)

file_summary = (
    file_summary
    .withColumn("year", F.regexp_extract("source_file", r"(\d{4})-\d{2}\.parquet", 1).cast("int"))
    .withColumn("month", F.regexp_extract("source_file", r"\d{4}-(\d{2})\.parquet", 1).cast("int"))
)

print("========== OVERVIEW ==========")
file_summary.select(
    F.count("*").alias("candidate_files"),
    F.sum(F.col("read_ok").cast("int")).alias("good_files"),
    F.sum((~F.col("read_ok")).cast("int")).alias("bad_files"),
    F.sum("n_rows").alias("total_rows"),
    F.min("min_time").alias("global_min_time"),
    F.max("max_time").alias("global_max_time")
).show(truncate=False)

print("========== ROWS BY YEAR ==========")
(
    file_summary.groupBy("year")
    .agg(
        F.sum("n_rows").alias("n_rows"),
        F.count("*").alias("n_files")
    )
    .orderBy("year")
    .show(20, truncate=False)
)

print("========== ROWS BY YEAR-MONTH ==========")
(
    file_summary.groupBy("year", "month")
    .agg(F.sum("n_rows").alias("n_rows"))
    .orderBy("year", "month")
    .show(100, truncate=False)
)



========== OVERVIEW ==========


+---------------+----------+---------+----------+-------------------+-------------------+
|candidate_files|good_files|bad_files|total_rows|global_min_time    |global_max_time    |
+---------------+----------+---------+----------+-------------------+-------------------+
|72             |72        |0        |223412046 |2001-01-01 00:03:14|2098-09-11 02:23:31|
+---------------+----------+---------+----------+-------------------+-------------------+

========== ROWS BY YEAR ==========


+----+--------+-------+
|year|n_rows  |n_files|
+----+--------+-------+
|2020|24649092|12     |
|2021|30904308|12     |
|2022|39656098|12     |
|2023|38310226|12     |
|2024|41169720|12     |
|2025|48722602|12     |
+----+--------+-------+

========== ROWS BY YEAR-MONTH ==========


+----+-----+-------+
|year|month|n_rows |
+----+-----+-------+
|2020|1    |6405008|
|2020|2    |6299367|
|2020|3    |3007687|
|2020|4    |238073 |
|2020|5    |348415 |
|2020|6    |549797 |
|2020|7    |800412 |
|2020|8    |1007286|
|2020|9    |1341017|
|2020|10   |1681132|
|2020|11   |1509000|
|2020|12   |1461898|
|2021|1    |1369769|
|2021|2    |1371709|
|2021|3    |1925152|
|2021|4    |2171187|
|2021|5    |2507109|
|2021|6    |2834264|
|2021|7    |2821746|
|2021|8    |2788757|
|2021|9    |2963793|
|2021|10   |3463504|
|2021|11   |3472949|
|2021|12   |3214369|
|2022|1    |2463931|
|2022|2    |2979431|
|2022|3    |3627882|
|2022|4    |3599920|
|2022|5    |3588295|
|2022|6    |3558124|
|2022|7    |3174394|
|2022|8    |3152677|
|2022|9    |3183767|
|2022|10   |3675411|
|2022|11   |3252717|
|2022|12   |3399549|
|2023|1    |3066766|
|2023|2    |2913955|
|2023|3    |3403766|
|2023|4    |3288250|
|2023|5    |3513649|
|2023|6    |3307234|
|2023|7    |2907108|
|2023|8    |2824209|
|2023|9    |2